<div style=" background-color: #940D0D;" >
<h1 style="margin: auto; padding: 20px 0; color:#fff; text-align: center">PROJET DATA ANALYST</h1>
<h2 style="margin: auto; padding: 20px 0; color:#fff; text-align: center">Produisez une étude de marché
</h2>
</div>

<div style="border-left: 6px solid #940D0D; background-color: #f9fafc; padding: 20px; border-radius: 10px;">

# Notebook 1 - Préparation des données

## Objective
Ce notebook prépare l'ensemble de données au niveau national pour le projet d'étude de marché international.

Les principaux objectifs sont les suivants :

- Charger tous les jeux de données bruts
- Examiner leur structure
- Sélectionner l’année de référence (2017)
- Nettoyer et normaliser les noms de pays
- Extraire les indicateurs pertinents
- Fusionner toutes les sources en un seul jeu de données par pays
- Identifier les valeurs manquantes
- Enregistrer un jeu de données intermédiaire propre pour les étapes suivantes

</div>

<div style="background-color:#940D0D ;" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Étape 1 - Charger tous les jeux de données bruts</h2>
</div>

<div style="border: 2px solid #940D0D;" >
<h3 style="margin: auto; padding: 20px; color: #224F59; ">1.1 - Import of bookstores</h3>
</div>

In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

<div style="border: 2px solid #940D0D;" >
<h3 style="margin: auto; padding: 20px; color: #224F59; ">1.2 - Chargement des fichiers CSV et Excel</h3>
</div>

In [12]:
food_path = "../data_raw/DisponibiliteAlimentaire_2017.csv"
pop_path = "../data_raw/Population_2000_2018.csv"
macro_path = "../data_raw/macro_eco_2000_2018.xlsx"
pol_path = "../data_raw/PoliticalStability.xlsx"
PIB_path = "../data_raw/PIB_EN.xlsx"
sous_alimentation_path = "../data_raw/sous_alimentation_2000_2018.xlsx"

In [13]:
df_food = pd.read_csv(food_path)
df_population = pd.read_csv(pop_path)
df_macro = pd.read_excel(macro_path)
df_pol = pd.read_excel(pol_path)
df_Pib = pd.read_excel(PIB_path)
df_under = pd.read_excel(sous_alimentation_path)

datasets = {
    "df_macro": df_macro,
    "df_Pib": df_Pib,
    "df_pol": df_pol,
    "df_under": df_under,
    "df_food": df_food,
    "df_population": df_population
}

for name, df in datasets.items():
    print("=" * 90)
    print(name.upper())
    print("shape:", df.shape)
    print("columns:")
    print(df.columns.tolist())
    print("\nhead:")
    display(df.head(3))

DF_MACRO
shape: (16008, 15)
columns:
['Code Domaine', 'Domaine', 'Code zone (FAO)', 'Zone', 'Code Élément', 'Élément', 'Code Produit', 'Produit', 'Code année', 'Année', 'Unité', 'Valeur', 'Symbole', 'Description du Symbole', 'Note']

head:


,Code Domaine,Domaine,Code zone (FAO),Zone,Code Élément,Élément,Code Produit,Produit,Code année,Année,Unité,Valeur,Symbole,Description du Symbole,Note
0,MK,Indicateurs macro,2,Afghanistan,6110,Valeur US $,22008,Produit Intérieur Brut,2000,2000,millions,3342.034168,X,Ciffre de sources internationales,NaN
1,MK,Indicateurs macro,2,Afghanistan,6119,Valeur US $ par habitant,22008,Produit Intérieur Brut,2000,2000,US$,171.009428,X,Ciffre de sources internationales,NaN
2,MK,Indicateurs macro,2,Afghanistan,6129,Croissance annuelle US$,22008,Produit Intérieur Brut,2000,2000,%,54.924825,X,Ciffre de sources internationales,NaN


DF_PIB
shape: (211, 15)
columns:
['Domain Code', 'Domain', 'Area Code (FAO)', 'Area', 'Element Code', 'Element', 'Item Code', 'Item', 'Year Code', 'Year', 'Unit', 'Value', 'Flag', 'Flag Description', 'Note']

head:


,Domain Code,Domain,Area Code (FAO),Area,Element Code,Element,Item Code,Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
0,MK,Macro Indicators,2,Afghanistan,6110,Value US$,22008,Gross Domestic Product,2018,2018,millions,"18,418.848",X,Figure from international organizations,NaN
1,MK,Macro Indicators,3,Albania,6110,Value US$,22008,Gross Domestic Product,2018,2018,millions,"15,156.440",X,Figure from international organizations,NaN
2,MK,Macro Indicators,4,Algeria,6110,Value US$,22008,Gross Domestic Product,2018,2018,millions,"174,910.895",X,Figure from international organizations,NaN


DF_POL
shape: (3526, 4)
columns:
['Country', 'Year', 'Political_Stability', 'Granularity']

head:


...

,Code Domaine,Domaine,Code zone (FAO),Zone,Code Élément,Élément,Code Produit,Produit,Code année,Année,Unité,Valeur,Symbole,Description du Symbole,Note
0,FS,Données de la sécurité alimentaire,2,Afghanistan,6121,Valeur,210041,Prévalence de la sous-alimentation (%) (moyenn...,20002002,2000-2002,%,47.8,E,Valeur estimée,NaN
1,FS,Données de la sécurité alimentaire,2,Afghanistan,6121,Valeur,210041,Prévalence de la sous-alimentation (%) (moyenn...,20012003,2001-2003,%,45.6,E,Valeur estimée,NaN
2,FS,Données de la sécurité alimentaire,2,Afghanistan,6121,Valeur,210041,Prévalence de la sous-alimentation (%) (moyenn...,20022004,2002-2004,%,40.6,E,Valeur estimée,NaN


DF_FOOD
shape: (176600, 14)
columns:
['Code Domaine', 'Domaine', 'Code zone', 'Zone', 'Code Élément', 'Élément', 'Code Produit', 'Produit', 'Code année', 'Année', 'Unité', 'Valeur', 'Symbole', 'Description du Symbole']

head:


,Code Domaine,Domaine,Code zone,Zone,Code Élément,Élément,Code Produit,Produit,Code année,Année,Unité,Valeur,Symbole,Description du Symbole
0,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5511,Production,2511,Blé et produits,2017,2017,Milliers de tonnes,"4,281.000",S,Données standardisées
1,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5611,Importations - Quantité,2511,Blé et produits,2017,2017,Milliers de tonnes,"2,302.000",S,Données standardisées
2,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5072,Variation de stock,2511,Blé et produits,2017,2017,Milliers de tonnes,-119.000,S,Données standardisées


DF_POPULATION
shape: (4411, 15)
columns:
['Code Domaine', 'Domaine', 'Code zone', 'Zone', 'Code Élément', 'Élément', 'Code Produit', 'Produit', 'Code année', 'Année', 'Unité', 'Valeur', 'Symbole', 'Description du Symbole', 'Note']

head:


,Code Domaine,Domaine,Code zone,Zone,Code Élément,Élément,Code Produit,Produit,Code année,Année,Unité,Valeur,Symbole,Description du Symbole,Note
0,OA,Séries temporelles annuelles,2,Afghanistan,511,Population totale,3010,Population-Estimations,2000,2000,1000 personnes,"20,779.953",X,Sources internationales sûres,NaN
1,OA,Séries temporelles annuelles,2,Afghanistan,511,Population totale,3010,Population-Estimations,2001,2001,1000 personnes,"21,606.988",X,Sources internationales sûres,NaN
2,OA,Séries temporelles annuelles,2,Afghanistan,511,Population totale,3010,Population-Estimations,2002,2002,1000 personnes,"22,600.770",X,Sources internationales sûres,NaN


<div style="background-color:#940D0D ;" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">fonctions utilitaires</h2>
</div>

In [14]:
#Normaliser le traitement des colonnes, des nombres et des pays.
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace("(", "", regex=False)
        .str.replace(")", "", regex=False)
        .str.replace("%", "pct", regex=False)
    )
    return df
    
#Les textes sont convertis en nombres même s'ils contiennent :
#une virgule française
#des espaces
#des caractères spéciaux

def to_numeric_series(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace("\u202f", "", regex=False)
        .str.replace("\xa0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce"
    )


def standardize_country_names(df, country_col="country", mapping=None):
    df = df.copy()
    df[country_col] = df[country_col].astype(str).str.strip()
    if mapping is not None:
        df[country_col] = df[country_col].replace(mapping)
    return df


def deduplicate_country(df):
    df = df.copy()
    return df.groupby("country", as_index=False).first()

In [15]:
food = clean_columns(df_food)
population = clean_columns(df_population)
macro = clean_columns(df_macro)
political = clean_columns(df_pol)
pib = clean_columns(df_Pib)
under = clean_columns(df_under)

for name, df_ in {
    "food": food,
    "population": population,
    "macro": macro,
    "political": political,
    "pib": pib,
    "under": under
}.items():
    print("=" * 90)
    print(name.upper())
    print(df_.columns.tolist())

FOOD
['code_domaine', 'domaine', 'code_zone', 'zone', 'code_élément', 'élément', 'code_produit', 'produit', 'code_année', 'année', 'unité', 'valeur', 'symbole', 'description_du_symbole']
POPULATION
['code_domaine', 'domaine', 'code_zone', 'zone', 'code_élément', 'élément', 'code_produit', 'produit', 'code_année', 'année', 'unité', 'valeur', 'symbole', 'description_du_symbole', 'note']
MACRO
['code_domaine', 'domaine', 'code_zone_fao', 'zone', 'code_élément', 'élément', 'code_produit', 'produit', 'code_année', 'année', 'unité', 'valeur', 'symbole', 'description_du_symbole', 'note']
POLITICAL
['country', 'year', 'political_stability', 'granularity']
PIB
['domain_code', 'domain', 'area_code_fao', 'area', 'element_code', 'element', 'item_code', 'item', 'year_code', 'year', 'unit', 'value', 'flag', 'flag_description', 'note']
UNDER
['code_domaine', 'domaine', 'code_zone_fao', 'zone', 'code_élément', 'élément', 'code_produit', 'produit', 'code_année', 'année', 'unité', 'valeur', 'symbole', '

<div style="border-left: 6px solid #940D0D; background-color: #f9fafc; padding: 20px; border-radius: 10px;">

# Ce que le résultat montre

La structure brute des données et les premières anomalies éventuelles.

## Utilité

Normaliser le traitement des colonnes, des nombres et des pays.


</div>

In [16]:
#S’assurer que l’année 2017 est bien disponible dans les différentes sources.
#Que 2017 peut être utilisée comme année de référence commune.

print("macro max year:", pd.to_numeric(macro["année"], errors="coerce").max())
print("population max year:", pd.to_numeric(population["année"], errors="coerce").max())

print(
    "political min/max year:",
    pd.to_numeric(political["year"], errors="coerce").min(),
    pd.to_numeric(political["year"], errors="coerce").max()
)

print("food unique years:", food["année"].unique())
print("under sample years:", under["année"].astype(str).unique()[:10])

macro max year: 2018
population max year: 2018
political min/max year: 2000 2018
food unique years: [2017]
under sample years: ['2000-2002' '2001-2003' '2002-2004' '2003-2005' '2004-2006' '2005-2007'
 '2006-2008' '2007-2009' '2008-2010' '2009-2011']


In [17]:
print("MACRO - unique elements")
display(macro["élément"].value_counts())

print("\nFOOD - top elements")
display(food["élément"].value_counts().head(20))

print("\nFOOD - top units")
display(food["unité"].value_counts().head(20))

print("\nFOOD - sample products")
display(food["produit"].value_counts().head(20))

MACRO - unique elements


élément
Valeur US $                             4004
Valeur US $ par habitant                4004
Croissance annuelle US$                 4000
Croissance annuelle US$ par habitant    4000
Name: count, dtype: int64


FOOD - top elements


élément
Disponibilité intérieure                                         15905
Importations - Quantité                                          15260
Disponibilité alimentaire en quantité (kg/personne/an)           14618
Disponibilité de matière grasse en quantité (g/personne/jour)    14512
Disponibilité de protéines en quantité (g/personne/jour)         14507
Nourriture                                                       14498
Disponibilité alimentaire (Kcal/personne/jour)                   14476
Résidus                                                          12567
Exportations - Quantité                                          12113
Variation de stock                                               11299
Production                                                       10334
Pertes                                                            5813
Alimentation pour touristes                                       5560
Autres utilisations (non alimentaire)                             529


FOOD - top units


unité
Milliers de tonnes    118487
g/personne/jour        29019
kg                     14618
Kcal/personne/jour     14476
Name: count, dtype: int64


FOOD - sample products


produit
Maïs et produits                   2593
Blé et produits                    2581
Pommes de Terre et produits        2486
Riz et produits                    2452
Lait - Excl Beurre                 2395
Oeufs                              2347
Légumineuses Autres et produits    2336
Orge et produits                   2301
Soja                               2222
Céréales, Autres                   2206
Légumes, Autres                    2204
Fruits, Autres                     2198
Arachides Decortiquees             2197
Edulcorants Autres                 2194
Haricots                           2141
Sucre Eq Brut                      2137
Graisses Animales Crue             2137
Feve de Cacao et produits          2088
Huil Plantes Oleif Autr            2083
Plantes Oleiferes, Autre           2077
Name: count, dtype: int64

<div style="border-left: 6px solid #940D0D; background-color: #f9fafc; padding: 20px; border-radius: 10px;">

## Utilité


Comprendre le contenu des fichiers :
- quels indicateurs existent
- quelles unités sont disponibles
- quels produits sont présents dans les données alimentaires



</div>

<div style="border: 2px solid #940D0D;" >
<h3 style="margin: auto; padding: 20px; color: #224F59; "> country_mapping </h3>
</div>

In [18]:
country_mapping = {
    "Afrique du Sud": "South Africa",
    "Albanie": "Albania",
    "Algérie": "Algeria",
    "Allemagne": "Germany",
    "Andorre": "Andorra",
    "Angola": "Angola",
    "Antigua-et-Barbuda": "Antigua and Barbuda",
    "Arabie saoudite": "Saudi Arabia",
    "Argentine": "Argentina",
    "Arménie": "Armenia",
    "Australie": "Australia",
    "Autriche": "Austria",
    "Azerbaïdjan": "Azerbaijan",
    "Bahreïn": "Bahrain",
    "Barbade": "Barbados",
    "Belgique": "Belgium",
    "Bhoutan": "Bhutan",
    "Bolivie (État plurinational de)": "Bolivia",
    "Bosnie-Herzégovine": "Bosnia and Herzegovina",
    "Brésil": "Brazil",
    "Bulgarie": "Bulgaria",
    "Cambodge": "Cambodia",
    "Cameroun": "Cameroon",
    "Chili": "Chile",
    "Chine, continentale": "China",
    "Colombie": "Colombia",
    "Corée du Sud": "South Korea",
    "Côte d'Ivoire": "Ivory Coast",
    "Croatie": "Croatia",
    "Danemark": "Denmark",
    "Égypte": "Egypt",
    "Émirats arabes unis": "United Arab Emirates",
    "Équateur": "Ecuador",
    "Espagne": "Spain",
    "États-Unis": "United States",
    "États-Unis d'Amérique": "United States",
    "Éthiopie": "Ethiopia",
    "Fédération de Russie": "Russia",
    "Finlande": "Finland",
    "Géorgie": "Georgia",
    "Grèce": "Greece",
    "Hongrie": "Hungary",
    "Inde": "India",
    "Indonésie": "Indonesia",
    "Irlande": "Ireland",
    "Islande": "Iceland",
    "Israël": "Israel",
    "Italie": "Italy",
    "Japon": "Japan",
    "Kazakhstan": "Kazakhstan",
    "Lettonie": "Latvia",
    "Lituanie": "Lithuania",
    "Luxembourg": "Luxembourg",
    "Malaisie": "Malaysia",
    "Maroc": "Morocco",
    "Mexique": "Mexico",
    "Nigéria": "Nigeria",
    "Norvège": "Norway",
    "Nouvelle-Zélande": "New Zealand",
    "Pays-Bas": "Netherlands",
    "Pérou": "Peru",
    "Pologne": "Poland",
    "Portugal": "Portugal",
    "République de Corée": "South Korea",
    "République démocratique du Congo": "DR Congo",
    "République dominicaine": "Dominican Republic",
    "République tchèque": "Czech Republic",
    "Roumanie": "Romania",
    "Royaume-Uni": "United Kingdom",
    "Singapour": "Singapore",
    "Slovaquie": "Slovakia",
    "Slovénie": "Slovenia",
    "Suède": "Sweden",
    "Suisse": "Switzerland",
    "Turquie": "Turkey",
    "Ukraine": "Ukraine",
    "Uruguay": "Uruguay",
    "Viet Nam": "Vietnam",
    "Vietnam": "Vietnam",
    "Zimbabwe": "Zimbabwe",
    "United States of America": "United States",
    "Russian Federation": "Russia",
    "Republic of Korea": "South Korea",
    "Iran (Islamic Republic of)": "Iran",
    "Syrian Arab Republic": "Syria",
    "Türkiye": "Turkey",
    "Venezuela (Bolivarian Republic of)": "Venezuela"
}

<div style="border-left: 6px solid #940D0D; background-color: #f9fafc; padding: 20px; border-radius: 10px;">

j'ai créé country_mapping pour convertir les noms de pays, qu'ils soient français ou autres, en un format unique et standardisé.

## Avantage

Cette étape est cruciale car différentes tables  utiliser :

- des noms français

- des noms anglais

- des formats longs et formels

- des formats abrégés

## Résultat souhaité

Obtenir un nom de pays uniforme dans tous les fichiers.

</div>

<div style="border: 2px solid #940D0D;" >
<h3 style="margin: auto; padding: 20px; color: #224F59; "> Données démographiques — chiffres de 2000 et 2017 et dynamique de la population </h3>
</div>

In [19]:
population = population.copy()

population["country_raw"] = population["zone"].astype(str).str.strip()
population["country"] = population["country_raw"].replace(country_mapping)

population["année_num"] = to_numeric_series(population["année"])
population["valeur_num"] = to_numeric_series(population["valeur"])

population_valid = population.dropna(subset=["country", "année_num", "valeur_num"]).copy()
population_valid["année_num"] = population_valid["année_num"].astype(int)

population_yearly = (
    population_valid
    .groupby(["country", "année_num"], as_index=False)["valeur_num"]
    .mean()
)

population_2000 = (
    population_yearly[population_yearly["année_num"] == 2000][["country", "valeur_num"]]
    .rename(columns={"valeur_num": "population_2000"})
    .copy()
)

population_2017 = (
    population_yearly[population_yearly["année_num"] == 2017][["country", "valeur_num"]]
    .rename(columns={"valeur_num": "population_2017"})
    .copy()
)

population_trend = population_2000.merge(population_2017, on="country", how="inner")
population_trend = population_trend[population_trend["population_2000"] > 0].copy()

population_trend["population_growth_2000_2017"] = (
    (population_trend["population_2017"] - population_trend["population_2000"])
    / population_trend["population_2000"]
)

population_trend["population_cagr_2000_2017"] = (
    (population_trend["population_2017"] / population_trend["population_2000"]) ** (1 / 17) - 1
)

print("population_trend shape:", population_trend.shape)
display(population_trend.head())

population_trend shape: (227, 5)


,country,population_2000,population_2017,population_growth_2000_2017,population_cagr_2000_2017
0,Afghanistan,"20,779.953","36,296.113",0.747,0.033
1,Albania,"3,129.243","2,884.169",-0.078,-0.005
2,Algeria,"31,042.235","41,389.189",0.333,0.017
3,Andorra,65.390,77.001,0.178,0.010
4,Angola,"16,395.473","29,816.766",0.819,0.036


In [20]:
print("Missing values:")
display(population_trend.isna().sum())

print("Describe:")
display(
    population_trend[[
        "population_2000",
        "population_2017",
        "population_growth_2000_2017",
        "population_cagr_2000_2017"
    ]].describe().T
)

Missing values:


country                        0
population_2000                0
population_2017                0
population_growth_2000_2017    0
population_cagr_2000_2017      0
dtype: int64

Describe:


,count,mean,std,min,25%,50%,75%,max
population_2000,227.000,"26,871.887","114,413.046",0.785,360.085,"4,362.187","15,846.497","1,290,550.765"
population_2017,227.000,"32,980.958","134,374.278",0.793,431.203,"5,447.900","20,391.001","1,421,021.791"
population_growth_2000_2017,227.000,0.293,0.369,-0.190,0.068,0.218,0.458,3.599
population_cagr_2000_2017,227.000,0.014,0.014,-0.012,0.004,0.012,0.022,0.094


<div style="border-left: 6px solid #940D0D; background-color: #f9fafc; padding: 20px; border-radius: 10px;">


- Nettoyage des données démographiques

- Extraction des données démographiques pour l'année 2000

- Extraction des données démographiques pour l'année 2017

### Création des fichiers suivants :

- population_growth_2000_2017

- population_cagr_2000_2017

### Avantage

Il s'agit d'un élément essentiel car il intègre la dimension démographique dynamique.

Comment procédons-nous ?

-  Je prends en compte la population pour chaque année et chaque pays.

-  Je construis les variables population_2000 et population_2017.

 Ensuite, Je calculs :

- La croissance relative

- Le taux de croissance annuel composé (TCAC)

Résultat souhaité :

- population_2017

- population_TCAC_2000_2017

Interprétation des résultats :

-  population_2017 = Taille actuelle du marché

-  population_TCAC_2000_2017 = Dynamique de la croissance démographique

</div>

<div style="border: 2px solid #940D0D;" >
<h3 style="margin: auto; padding: 20px; color: #224F59; "> Macro d'extraction 2017 </h3>
</div>

In [21]:
#Extraction des données économiques de 2017 en guise de prélude à l'élaboration d'indicateurs du GDP.
macro["année_num"] = to_numeric_series(macro["année"])
macro["valeur_num"] = to_numeric_series(macro["valeur"])

macro["country_raw"] = macro["zone"].astype(str).str.strip()
macro["country"] = macro["country_raw"].replace(country_mapping)

macro_2017 = macro[macro["année_num"] == 2017].copy()

print("macro_2017 shape:", macro_2017.shape)
print("countries:", macro_2017["country"].nunique())
display(macro_2017["élément"].value_counts())

macro_2017 shape: (844, 19)
countries: 211


élément
Valeur US $                             211
Valeur US $ par habitant                211
Croissance annuelle US$                 211
Croissance annuelle US$ par habitant    211
Name: count, dtype: int64

<div style="border: 2px solid #940D0D;" >
<h3 style="margin: auto; padding: 20px; color: #224F59; ">  Indicateurs du GDP</h3>
</div>

In [22]:
gdp_2017 = (
    macro_2017[macro_2017["élément"] == "Valeur US $"][["country", "valeur_num"]]
    .rename(columns={"valeur_num": "gdp_usd_2017"})
    .copy()
)

gdp_pc_2017 = (
    macro_2017[macro_2017["élément"] == "Valeur US $ par habitant"][["country", "valeur_num"]]
    .rename(columns={"valeur_num": "gdp_per_capita_2017"})
    .copy()
)

gdp_growth_2017 = (
    macro_2017[macro_2017["élément"] == "Croissance annuelle US$"][["country", "valeur_num"]]
    .rename(columns={"valeur_num": "gdp_growth_2017"})
    .copy()
)

gdp_pc_growth_2017 = (
    macro_2017[macro_2017["élément"] == "Croissance annuelle US$ par habitant"][["country", "valeur_num"]]
    .rename(columns={"valeur_num": "gdp_pc_growth_2017"})
    .copy()
)

display(gdp_2017.head())
display(gdp_pc_2017.head())
display(gdp_growth_2017.head())
display(gdp_pc_growth_2017.head())

,country,gdp_usd_2017
68,Afghanistan,"18,896.352"
144,South Africa,"380,851.444"
220,Albania,"13,019.730"
296,Algeria,"170,096.987"
372,Germany,"3,690,849.153"


,country,gdp_per_capita_2017
69,Afghanistan,530.150
145,South Africa,"6,723.929"
221,Albania,"4,521.752"
297,Algeria,"4,134.936"
373,Germany,"44,670.222"


,country,gdp_growth_2017
70,Afghanistan,4.866
146,South Africa,17.704
222,Albania,9.766
298,Algeria,6.288
374,Germany,6.441


,country,gdp_pc_growth_2017
71,Afghanistan,1.902
147,South Africa,17.249
223,Albania,9.831
299,Algeria,4.228
375,Germany,6.064


<div style="border-left: 6px solid #940D0D; background-color: #f9fafc; padding: 20px; border-radius: 10px;">

Dans cette section, j'extrais les données suivantes :

- gdp_USD_2017

- gdp_par_habitant_2017

- Croissance_du_gdp_2017

- Croissance_du_gdp_par_habitant_2017

### Intérêt

Ces indicateurs représentent la principale dimension économique du projet.

Comment interpréter chaque variable ?

- gdp_USD_2017 : Taille de la macroéconomie

- gdp_par_habitant_2017 : Niveau de revenu par habitant

- Croissance_du_gdp_2017 : Croissance économique globale

- Croissance_du_gdp_par_habitant_2017 : Croissance par habitant

</div>

<div style="border: 2px solid #940D0D;" >
<h3 style="margin: auto; padding: 20px; color: #224F59; "> Political stability 2017 </h3>
</div>

In [23]:
political["country"] = political["country"].astype(str).str.strip().replace(country_mapping)
political["year"] = to_numeric_series(political["year"])
political["political_stability"] = to_numeric_series(political["political_stability"])

political_2017 = (
    political[political["year"] == 2017][["country", "political_stability"]]
    .rename(columns={"political_stability": "political_stability_2017"})
    .copy()
)

print("political_2017 shape:", political_2017.shape)
print("countries:", political_2017["country"].nunique())
display(political_2017.head())

political_2017 shape: (198, 2)
countries: 198


,country,political_stability_2017
16,Afghanistan,-2.800
34,Albania,0.380
52,Algeria,-0.920
67,American Samoa,1.220
85,Andorra,1.420


<div style="border-left: 6px solid #940D0D; background-color: #f9fafc; padding: 20px; border-radius: 10px;">

j'extrais :

- political_stability_2017

Avantage

Ajout de la dimension institutionnelle et des risques politiques.


Cette variable est essentielle car l’attractivité du marché n’est pas uniquement économique, mais aussi liée au degré de stabilité.

</div>

<div style="border: 2px solid #940D0D;" >
<h3 style="margin: auto; padding: 20px; color: #224F59; "> Food data 2017 </h3>
</div>

In [24]:
#J'ai préparé le fichier alimentaire pour 2017. J'ai isolé l'année de référence et créé une base 
#de données prête à recevoir les indicateurs nutritionnels.
#Données nécessaires à l'extraction :
#• Calories
#• Protéines
#• Volaille

food["country_raw"] = food["zone"].astype(str).str.strip()
food["country"] = food["country_raw"].replace(country_mapping)

food["valeur_num"] = to_numeric_series(food["valeur"])
food["année_num"] = to_numeric_series(food["année"])

food_2017 = food[food["année_num"] == 2017].copy()

print("food_2017 shape:", food_2017.shape)
print("countries:", food_2017["country"].nunique())
display(food_2017["élément"].value_counts().head(15))

food_2017 shape: (176600, 18)
countries: 174


élément
Disponibilité intérieure                                         15905
Importations - Quantité                                          15260
Disponibilité alimentaire en quantité (kg/personne/an)           14618
Disponibilité de matière grasse en quantité (g/personne/jour)    14512
Disponibilité de protéines en quantité (g/personne/jour)         14507
Nourriture                                                       14498
Disponibilité alimentaire (Kcal/personne/jour)                   14476
Résidus                                                          12567
Exportations - Quantité                                          12113
Variation de stock                                               11299
Production                                                       10334
Pertes                                                            5813
Alimentation pour touristes                                       5560
Autres utilisations (non alimentaire)                             529

<div style="border: 2px solid #940D0D;" >
<h3 style="margin: auto; padding: 20px; color: #224F59; "> Food indicators </h3>
</div>

In [25]:

food_kcal_2017 = (
    food_2017[food_2017["élément"] == "Disponibilité alimentaire (Kcal/personne/jour)"]
    .groupby("country", as_index=False)["valeur_num"]
    .sum()
    .rename(columns={"valeur_num": "food_kcal_per_capita_day_2017"})
)

food_protein_2017 = (
    food_2017[food_2017["élément"] == "Disponibilité de protéines en quantité (g/personne/jour)"]
    .groupby("country", as_index=False)["valeur_num"]
    .sum()
    .rename(columns={"valeur_num": "food_protein_g_per_capita_day_2017"})
)

display(food_kcal_2017.head())
display(food_protein_2017.head())

print("kcal countries:", food_kcal_2017["country"].nunique())
print("protein countries:", food_protein_2017["country"].nunique())

,country,food_kcal_per_capita_day_2017
0,Afghanistan,"1,997.000"
1,Albania,"3,400.000"
2,Algeria,"3,345.000"
3,Angola,"2,266.000"
4,Antigua and Barbuda,"2,429.000"


,country,food_protein_g_per_capita_day_2017
0,Afghanistan,54.090
1,Albania,119.500
2,Algeria,92.850
3,Angola,54.090
4,Antigua and Barbuda,81.150


kcal countries: 172
protein countries: 172


<div style="border-left: 6px solid #940D0D; background-color: #f9fafc; padding: 20px; border-radius: 10px;">
j'extrais :

- food_kcal_per_capita_day_2017

- food_protein_g_per_capita_day_2017

Ajout de la dimension nutritionnelle globale à l’étude.

Résultat ?

- food_kcal : disponibilité nutritionnelle

- food_protein : disponibilité en protéines

Ceci est important car le projet porte sur l’alimentation humaine et animale, et notamment la volaille.
</div>

<div style="border: 2px solid #940D0D;" >
<h3 style="margin: auto; padding: 20px; color: #224F59; "> Optional poultry indicators </h3>
</div>


In [26]:
# New variable 1: poultry food availability
poultry_kcal_2017 = (
    food_2017[
        (food_2017["élément"] == "Disponibilité alimentaire (Kcal/personne/jour)") &
        (food_2017["produit"] == "Viande de Volailles")
    ]
    .groupby("country", as_index=False)["valeur_num"]
    .sum()
    .rename(columns={"valeur_num": "poultry_kcal_per_capita_day_2017"})
)

# New variable 2: poultry imports
poultry_import_2017 = (
    food_2017[
        (food_2017["élément"] == "Importations - Quantité") &
        (food_2017["produit"] == "Viande de Volailles")
    ]
    .groupby("country", as_index=False)["valeur_num"]
    .sum()
    .rename(columns={"valeur_num": "poultry_import_1000_tonnes_2017"})
)

display(poultry_kcal_2017.head())
display(poultry_import_2017.head())

,country,poultry_kcal_per_capita_day_2017
0,Afghanistan,5.000
1,Albania,85.000
2,Algeria,22.000
3,Angola,35.000
4,Antigua and Barbuda,233.000


,country,poultry_import_1000_tonnes_2017
0,Afghanistan,29.000
1,Albania,38.000
2,Algeria,2.000
3,Angola,277.000
4,Antigua and Barbuda,7.000


<div style="border-left: 6px solid #940D0D; background-color: #f9fafc; padding: 20px; border-radius: 10px;">


 j'ai ajouté :

- poultry_kcal_per_capita_day_2017

- poultry_import_1000_tonnes_2017

c'est essentiel car il établit un lien direct entre l'étude et le produit de l'entreprise.

- poultry_kcal_per_capita_day_2017
Mesure la disponibilité des calories provenant de la viande de volaille dans l'alimentation.

- poultry_import_1000_tonnes_2017
Mesure les importations de volaille.
</div>

In [27]:
#Examiner une variable supplémentaire possible.

under["country_raw"] = under["zone"].astype(str).str.strip()
under["country"] = under["country_raw"].replace(country_mapping)

under["valeur_num"] = to_numeric_series(under["valeur"])
under["année_str"] = under["année"].astype(str)

under_2015_2017 = (
    under[under["année_str"] == "2015-2017"][["country", "valeur_num"]]
    .rename(columns={"valeur_num": "undernourishment_2015_2017"})
    .copy()
)

print("under_2015_2017 shape:", under_2015_2017.shape)
print("countries with usable numeric values:", under_2015_2017["undernourishment_2015_2017"].notna().sum())
display(under_2015_2017.head(20))

under_2015_2017 shape: (204, 2)
countries with usable numeric values: 18


,country,undernourishment_2015_2017
15,Afghanistan,NaN
33,South Africa,NaN
51,Albania,NaN
69,Algeria,NaN
87,Germany,NaN
105,Andorra,NaN
123,Angola,NaN
141,Antigua and Barbuda,NaN
159,Saudi Arabia,NaN
177,Argentina,NaN


<div style="border: 2px solid #940D0D;" >
<h3 style="margin: auto; padding: 20px; color: #224F59; "> Dédupliquer toutes les tables au niveau du pays </h3>
</div>


In [28]:
#Appliquez la règle `deduplicate_country` à toutes les tables.
#Avant la fusion, assurez-vous que chaque pays n'apparaît qu'une seule fois dans chaque table.

population_trend = deduplicate_country(population_trend)
gdp_2017 = deduplicate_country(gdp_2017)
gdp_pc_2017 = deduplicate_country(gdp_pc_2017)
gdp_growth_2017 = deduplicate_country(gdp_growth_2017)
gdp_pc_growth_2017 = deduplicate_country(gdp_pc_growth_2017)
political_2017 = deduplicate_country(political_2017)
food_kcal_2017 = deduplicate_country(food_kcal_2017)
food_protein_2017 = deduplicate_country(food_protein_2017)
poultry_kcal_2017 = deduplicate_country(poultry_kcal_2017)
poultry_import_2017 = deduplicate_country(poultry_import_2017)
under_2015_2017 = deduplicate_country(under_2015_2017)

In [29]:
#On compte le nombre de pays dans chaque tableau.
#Afin de déterminer :
#• Quels tableaux couvrent le plus grand nombre de pays ?
#• Quels tableaux, une fois combinés, pourraient présenter le plus grand nombre de pays manquants ?


country_counts = {
    "population_trend": population_trend["country"].nunique(),
    "gdp": gdp_2017["country"].nunique(),
    "gdp_per_capita": gdp_pc_2017["country"].nunique(),
    "gdp_growth": gdp_growth_2017["country"].nunique(),
    "gdp_pc_growth": gdp_pc_growth_2017["country"].nunique(),
    "political": political_2017["country"].nunique(),
    "food_kcal": food_kcal_2017["country"].nunique(),
    "food_protein": food_protein_2017["country"].nunique(),
    "poultry_kcal": poultry_kcal_2017["country"].nunique(),
    "poultry_import": poultry_import_2017["country"].nunique(),
    "under_optional": under_2015_2017["country"].nunique(),
}

pd.Series(country_counts).sort_values(ascending=False)

population_trend    227
gdp                 211
gdp_per_capita      211
gdp_growth          211
gdp_pc_growth       211
under_optional      204
political           198
food_kcal           172
food_protein        172
poultry_kcal        172
poultry_import      170
dtype: int64

<div style="background-color:#940D0D ;" >
<h2 style="margin: auto; padding: 20px; color:#fff; "> Fusionner l'ensemble de données principal </h2>
</div>


In [30]:
df_merged = population_trend.copy()

for df_part in [
    gdp_2017,
    gdp_pc_2017,
    gdp_growth_2017,
    gdp_pc_growth_2017,
    political_2017,
    food_kcal_2017,
    food_protein_2017,
    poultry_kcal_2017,
    poultry_import_2017
]:
    df_merged = df_merged.merge(df_part, on="country", how="outer")

df_merged = df_merged.drop_duplicates(subset=["country"]).copy()

print("Merged shape:", df_merged.shape)
print("Countries:", df_merged["country"].nunique())
display(df_merged.head())

Merged shape: (309, 14)
Countries: 309


,country,population_2000,population_2017,population_growth_2000_2017,population_cagr_2000_2017,gdp_usd_2017,gdp_per_capita_2017,gdp_growth_2017,gdp_pc_growth_2017,political_stability_2017,food_kcal_per_capita_day_2017,food_protein_g_per_capita_day_2017,poultry_kcal_per_capita_day_2017,poultry_import_1000_tonnes_2017
0,Afghanistan,"20,779.953","36,296.113",0.747,0.033,"18,896.352",530.150,4.866,1.902,-2.800,"1,997.000",54.090,5.000,29.000
1,Albania,"3,129.243","2,884.169",-0.078,-0.005,"13,019.730","4,521.752",9.766,9.831,0.380,"3,400.000",119.500,85.000,38.000
2,Algeria,"31,042.235","41,389.189",0.333,0.017,"170,096.987","4,134.936",6.288,4.228,-0.920,"3,345.000",92.850,22.000,2.000
3,American Samoa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.220,NaN,NaN,NaN,NaN
4,Andorra,65.390,77.001,0.178,0.010,"3,000.160","40,632.206",3.645,1.825,1.420,NaN,NaN,NaN,NaN


In [31]:
df_merged_with_under = df_merged.merge(under_2015_2017, on="country", how="left")

print("Merged with undernourishment shape:", df_merged_with_under.shape)
display(df_merged_with_under.head())

Merged with undernourishment shape: (309, 15)


,country,population_2000,population_2017,population_growth_2000_2017,population_cagr_2000_2017,gdp_usd_2017,gdp_per_capita_2017,gdp_growth_2017,gdp_pc_growth_2017,political_stability_2017,food_kcal_per_capita_day_2017,food_protein_g_per_capita_day_2017,poultry_kcal_per_capita_day_2017,poultry_import_1000_tonnes_2017,undernourishment_2015_2017
0,Afghanistan,"20,779.953","36,296.113",0.747,0.033,"18,896.352",530.150,4.866,1.902,-2.800,"1,997.000",54.090,5.000,29.000,NaN
1,Albania,"3,129.243","2,884.169",-0.078,-0.005,"13,019.730","4,521.752",9.766,9.831,0.380,"3,400.000",119.500,85.000,38.000,NaN
2,Algeria,"31,042.235","41,389.189",0.333,0.017,"170,096.987","4,134.936",6.288,4.228,-0.920,"3,345.000",92.850,22.000,2.000,NaN
3,American Samoa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.220,NaN,NaN,NaN,NaN,NaN
4,Andorra,65.390,77.001,0.178,0.010,"3,000.160","40,632.206",3.645,1.825,1.420,NaN,NaN,NaN,NaN,NaN


<div style="border-left: 6px solid #940D0D; background-color: #f9fafc; padding: 20px; border-radius: 10px;">


Données combinées :

- population

- economy

- political stability

- food

- poultry variables

Construire un tableau unifié au niveau national.

Résultat : Un tableau :

- Chaque ligne = pays

- Chaque colonne = variable
</div>

In [32]:
#Conversion numérique après fusion
#Après la fusion, nous convertissons toutes les colonnes au format numérique.
#Nous nous assurons que toutes les colonnes sont bien numériques avant toute analyse ultérieure.

numeric_cols = [col for col in df_merged.columns if col != "country"]
for col in numeric_cols:
    df_merged[col] = pd.to_numeric(df_merged[col], errors="coerce")

numeric_cols_with_under = [col for col in df_merged_with_under.columns if col != "country"]
for col in numeric_cols_with_under:
    df_merged_with_under[col] = pd.to_numeric(df_merged_with_under[col], errors="coerce")

In [33]:
print("Main merged dataset info")
df_merged.info()

print("\nSummary statistics")
display(df_merged.describe().T)

Main merged dataset info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 309 entries, 0 to 308
Data columns (total 14 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   country                             309 non-null    object 
 1   population_2000                     227 non-null    float64
 2   population_2017                     227 non-null    float64
 3   population_growth_2000_2017         227 non-null    float64
 4   population_cagr_2000_2017           227 non-null    float64
 5   gdp_usd_2017                        211 non-null    float64
 6   gdp_per_capita_2017                 211 non-null    float64
 7   gdp_growth_2017                     211 non-null    float64
 8   gdp_pc_growth_2017                  211 non-null    float64
 9   political_stability_2017            188 non-null    float64
 10  food_kcal_per_capita_day_2017       172 non-null    float64
 11  food_protein_g_per_c

,count,mean,std,min,25%,50%,75%,max
population_2000,227.000,"26,871.887","114,413.046",0.785,360.085,"4,362.187","15,846.497","1,290,550.765"
population_2017,227.000,"32,980.958","134,374.278",0.793,431.203,"5,447.900","20,391.001","1,421,021.791"
population_growth_2000_2017,227.000,0.293,0.369,-0.190,0.068,0.218,0.458,3.599
population_cagr_2000_2017,227.000,0.014,0.014,-0.012,0.004,0.012,0.022,0.094
gdp_usd_2017,211.000,"442,902.776","1,875,352.790",45.296,"5,721.181","24,979.190","191,176.476","19,477,337.000"
gdp_per_capita_2017,211.000,"16,886.035","25,550.734",295.621,"2,082.324","6,450.320","19,806.127","173,611.815"
gdp_growth_2017,211.000,7.100,7.539,-27.928,4.502,7.272,9.941,36.133
gdp_pc_growth_2017,211.000,5.866,7.472,-29.218,3.383,6.002,8.921,32.525
political_stability_2017,188.000,-0.124,0.981,-2.940,-0.692,-0.045,0.650,1.920
food_kcal_per_capita_day_2017,172.000,"2,858.395",454.544,"1,754.000","2,514.250","2,871.500","3,250.250","3,770.000"


<div style="border: 2px solid #940D0D;" >
<h3 style="margin: auto; padding: 20px; color: #224F59; ">Informations et résumé sur l'ensemble de données </h3>
</div>


In [34]:
missing_summary = (
    df_merged.isna()
    .sum()
    .to_frame("missing_count")
    .sort_values("missing_count", ascending=False)
)

missing_summary["missing_pct"] = missing_summary["missing_count"] / len(df_merged) * 100
missing_summary

,missing_count,missing_pct
poultry_import_1000_tonnes_2017,139,44.984
food_kcal_per_capita_day_2017,137,44.337
food_protein_g_per_capita_day_2017,137,44.337
poultry_kcal_per_capita_day_2017,137,44.337
political_stability_2017,121,39.159
gdp_usd_2017,98,31.715
gdp_per_capita_2017,98,31.715
gdp_growth_2017,98,31.715
gdp_pc_growth_2017,98,31.715
population_2000,82,26.537


In [35]:
missing_summary_under = (
    df_merged_with_under.isna()
    .sum()
    .to_frame("missing_count")
    .sort_values("missing_count", ascending=False)
)

missing_summary_under["missing_pct"] = missing_summary_under["missing_count"] / len(df_merged_with_under) * 100
missing_summary_under

,missing_count,missing_pct
undernourishment_2015_2017,291,94.175
poultry_import_1000_tonnes_2017,139,44.984
food_kcal_per_capita_day_2017,137,44.337
food_protein_g_per_capita_day_2017,137,44.337
poultry_kcal_per_capita_day_2017,137,44.337
political_stability_2017,121,39.159
gdp_usd_2017,98,31.715
gdp_per_capita_2017,98,31.715
gdp_growth_2017,98,31.715
gdp_pc_growth_2017,98,31.715


<div style="background-color:#940D0D ;" >
<h2 style="margin: auto; padding: 20px; color:#fff; "> Constituer un ensemble de données fonctionnel </h2>
</div>


In [36]:
# les variables de la modélisation
main_vars = [
    "population_2017",
    "population_cagr_2000_2017",
    "gdp_per_capita_2017",
    "gdp_pc_growth_2017",
    "political_stability_2017",
    "food_kcal_per_capita_day_2017",
    "food_protein_g_per_capita_day_2017",
    "poultry_kcal_per_capita_day_2017",
    "poultry_import_1000_tonnes_2017",
]

# Version propre
df_working_clean = df_merged.dropna(subset=main_vars).copy()

print("Full merged dataset:", df_merged.shape)
print("Working clean dataset:", df_working_clean.shape)
print("Number of countries in working clean dataset:", df_working_clean["country"].nunique())

display(df_working_clean.head())

Full merged dataset: (309, 14)
Working clean dataset: (105, 14)
Number of countries in working clean dataset: 105


,country,population_2000,population_2017,population_growth_2000_2017,population_cagr_2000_2017,gdp_usd_2017,gdp_per_capita_2017,gdp_growth_2017,gdp_pc_growth_2017,political_stability_2017,food_kcal_per_capita_day_2017,food_protein_g_per_capita_day_2017,poultry_kcal_per_capita_day_2017,poultry_import_1000_tonnes_2017
0,Afghanistan,"20,779.953","36,296.113",0.747,0.033,"18,896.352",530.150,4.866,1.902,-2.800,"1,997.000",54.090,5.000,29.000
1,Albania,"3,129.243","2,884.169",-0.078,-0.005,"13,019.730","4,521.752",9.766,9.831,0.380,"3,400.000",119.500,85.000,38.000
2,Algeria,"31,042.235","41,389.189",0.333,0.017,"170,096.987","4,134.936",6.288,4.228,-0.920,"3,345.000",92.850,22.000,2.000
5,Angola,"16,395.473","29,816.766",0.819,0.036,"122,123.859","4,042.681",20.767,16.553,-0.330,"2,266.000",54.090,35.000,277.000
7,Antigua and Barbuda,76.016,95.426,0.255,0.013,"1,467.978","16,110.556",2.185,1.563,0.750,"2,429.000",81.150,233.000,7.000


In [49]:
display(df_working_clean.head().T)

,0,1,2,5,7
country,Afghanistan,Albania,Algeria,Angola,Antigua and Barbuda
population_2000,"20,779.953","3,129.243","31,042.235","16,395.473",76.016
population_2017,"36,296.113","2,884.169","41,389.189","29,816.766",95.426
population_growth_2000_2017,0.747,-0.078,0.333,0.819,0.255
population_cagr_2000_2017,0.033,-0.005,0.017,0.036,0.013
gdp_usd_2017,"18,896.352","13,019.730","170,096.987","122,123.859","1,467.978"
gdp_per_capita_2017,530.150,"4,521.752","4,134.936","4,042.681","16,110.556"
gdp_growth_2017,4.866,9.766,6.288,20.767,2.185
gdp_pc_growth_2017,1.902,9.831,4.228,16.553,1.563
political_stability_2017,-2.800,0.380,-0.920,-0.330,0.750


In [37]:
print(df_working_clean.shape)
display(df_working_clean.head(20))

(105, 14)


,country,population_2000,population_2017,population_growth_2000_2017,population_cagr_2000_2017,gdp_usd_2017,gdp_per_capita_2017,gdp_growth_2017,gdp_pc_growth_2017,political_stability_2017,food_kcal_per_capita_day_2017,food_protein_g_per_capita_day_2017,poultry_kcal_per_capita_day_2017,poultry_import_1000_tonnes_2017
0,Afghanistan,"20,779.953","36,296.113",0.747,0.033,"18,896.352",530.150,4.866,1.902,-2.800,"1,997.000",54.090,5.000,29.000
1,Albania,"3,129.243","2,884.169",-0.078,-0.005,"13,019.730","4,521.752",9.766,9.831,0.380,"3,400.000",119.500,85.000,38.000
2,Algeria,"31,042.235","41,389.189",0.333,0.017,"170,096.987","4,134.936",6.288,4.228,-0.920,"3,345.000",92.850,22.000,2.000
5,Angola,"16,395.473","29,816.766",0.819,0.036,"122,123.859","4,042.681",20.767,16.553,-0.330,"2,266.000",54.090,35.000,277.000
7,Antigua and Barbuda,76.016,95.426,0.255,0.013,"1,467.978","16,110.556",2.185,1.563,0.750,"2,429.000",81.150,233.000,7.000
9,Argentina,"36,870.787","43,937.140",0.192,0.010,"643,628.396","14,609.783",15.442,14.430,0.170,"3,239.000",102.660,182.000,8.000
10,Armenia,"3,069.591","2,944.791",-0.041,-0.002,"11,527.459","4,041.995",9.305,9.838,-0.620,"3,072.000",97.330,54.000,35.000
12,Australia,"18,991.431","24,584.620",0.295,0.015,"1,412,242.647","57,430.804",8.013,6.280,0.890,"3,307.000",108.010,192.000,16.000
14,Azerbaijan,"8,122.741","9,845.320",0.212,0.011,"40,866.632","4,057.624",7.921,6.900,-0.750,"3,102.000",92.300,44.000,27.000
15,Bahamas,298.051,381.755,0.281,0.015,"12,357.600","30,969.876",4.419,3.623,0.990,"2,043.000",61.370,182.000,24.000


In [51]:
display(df_working_clean.head().T)

,0,1,2,5,7
country,Afghanistan,Albania,Algeria,Angola,Antigua and Barbuda
population_2000,"20,779.953","3,129.243","31,042.235","16,395.473",76.016
population_2017,"36,296.113","2,884.169","41,389.189","29,816.766",95.426
population_growth_2000_2017,0.747,-0.078,0.333,0.819,0.255
population_cagr_2000_2017,0.033,-0.005,0.017,0.036,0.013
gdp_usd_2017,"18,896.352","13,019.730","170,096.987","122,123.859","1,467.978"
gdp_per_capita_2017,530.150,"4,521.752","4,134.936","4,042.681","16,110.556"
gdp_growth_2017,4.866,9.766,6.288,20.767,2.185
gdp_pc_growth_2017,1.902,9.831,4.228,16.553,1.563
political_stability_2017,-2.800,0.380,-0.920,-0.330,0.750


In [38]:
df_working_clean.isna().sum()

country                               0
population_2000                       0
population_2017                       0
population_growth_2000_2017           0
population_cagr_2000_2017             0
gdp_usd_2017                          0
gdp_per_capita_2017                   0
gdp_growth_2017                       0
gdp_pc_growth_2017                    0
political_stability_2017              0
food_kcal_per_capita_day_2017         0
food_protein_g_per_capita_day_2017    0
poultry_kcal_per_capita_day_2017      0
poultry_import_1000_tonnes_2017       0
dtype: int64

In [39]:
df_working_clean["country"].duplicated().sum()

0

In [40]:
df_working_clean.columns.tolist()

['country',
 'population_2000',
 'population_2017',
 'population_growth_2000_2017',
 'population_cagr_2000_2017',
 'gdp_usd_2017',
 'gdp_per_capita_2017',
 'gdp_growth_2017',
 'gdp_pc_growth_2017',
 'political_stability_2017',
 'food_kcal_per_capita_day_2017',
 'food_protein_g_per_capita_day_2017',
 'poultry_kcal_per_capita_day_2017',
 'poultry_import_1000_tonnes_2017']

<div style="border: 2px solid #940D0D;" >
<h3 style="margin: auto; padding: 20px; color: #224F59; "> Pays exclus </h3>
</div>



In [41]:
excluded = df_merged[~df_merged["country"].isin(df_working_clean["country"])].copy()
print("Excluded countries:", excluded["country"].nunique())
display(excluded.head(20))

Excluded countries: 204


,country,population_2000,population_2017,population_growth_2000_2017,population_cagr_2000_2017,gdp_usd_2017,gdp_per_capita_2017,gdp_growth_2017,gdp_pc_growth_2017,political_stability_2017,food_kcal_per_capita_day_2017,food_protein_g_per_capita_day_2017,poultry_kcal_per_capita_day_2017,poultry_import_1000_tonnes_2017
3,American Samoa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.220,NaN,NaN,NaN,NaN
4,Andorra,65.390,77.001,0.178,0.010,"3,000.160","40,632.206",3.645,1.825,1.420,NaN,NaN,NaN,NaN
6,Anguilla,11.252,14.584,0.296,0.015,281.056,"18,718.319",-11.826,-13.012,NaN,NaN,NaN,NaN,NaN
8,Antilles néerlandaises (ex),215.459,275.186,0.277,0.014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,Aruba,90.853,105.366,0.160,0.009,"3,092.430","29,329.092",3.646,3.091,NaN,NaN,NaN,NaN,NaN
13,Austria,"8,069.276","8,819.901",0.093,0.005,"417,261.152","47,429.536",5.484,4.752,NaN,"3,694.000",108.110,65.000,110.000
16,Bahrain,664.611,"1,494.076",1.248,0.049,"35,473.670","24,349.837",10.047,6.483,-0.960,NaN,NaN,NaN,NaN
19,Belarus,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.050,NaN,NaN,NaN,NaN
22,Benin,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.030,NaN,NaN,NaN,NaN
23,Bermuda,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000,NaN,NaN,NaN,NaN


In [42]:
# Version imputée : on conserve plus de pays
impute_vars = [
    "population_2017",
    "population_cagr_2000_2017",
    "gdp_per_capita_2017",
    "gdp_pc_growth_2017",
    "political_stability_2017",
    "food_kcal_per_capita_day_2017",
    "food_protein_g_per_capita_day_2017",
    "poultry_kcal_per_capita_day_2017",
    "poultry_import_1000_tonnes_2017",
]

df_working_imputed = df_merged[["country"] + impute_vars].copy()

for col in impute_vars:
    median_value = df_working_imputed[col].median()
    df_working_imputed[col] = df_working_imputed[col].fillna(median_value)

print("Working imputed dataset:", df_working_imputed.shape)
print("Number of countries in imputed dataset:", df_working_imputed["country"].nunique())

display(df_working_imputed.head())

Working imputed dataset: (309, 10)
Number of countries in imputed dataset: 309


,country,population_2017,population_cagr_2000_2017,gdp_per_capita_2017,gdp_pc_growth_2017,political_stability_2017,food_kcal_per_capita_day_2017,food_protein_g_per_capita_day_2017,poultry_kcal_per_capita_day_2017,poultry_import_1000_tonnes_2017
0,Afghanistan,"36,296.113",0.033,530.150,1.902,-2.800,"1,997.000",54.090,5.000,29.000
1,Albania,"2,884.169",-0.005,"4,521.752",9.831,0.380,"3,400.000",119.500,85.000,38.000
2,Algeria,"41,389.189",0.017,"4,134.936",4.228,-0.920,"3,345.000",92.850,22.000,2.000
3,American Samoa,"5,447.900",0.012,"6,450.320",6.002,1.220,"2,871.500",81.390,64.000,16.000
4,Andorra,77.001,0.010,"40,632.206",1.825,1.420,"2,871.500",81.390,64.000,16.000


In [43]:
print("Missing values after imputation:")
display(df_working_imputed.isna().sum())

Missing values after imputation:


country                               0
population_2017                       0
population_cagr_2000_2017             0
gdp_per_capita_2017                   0
gdp_pc_growth_2017                    0
political_stability_2017              0
food_kcal_per_capita_day_2017         0
food_protein_g_per_capita_day_2017    0
poultry_kcal_per_capita_day_2017      0
poultry_import_1000_tonnes_2017       0
dtype: int64

In [44]:
# On construit les variables dérivées sur la version imputée
df_final = df_working_imputed.copy()

df_final["protein_kcal_ratio"] = (
    df_final["food_protein_g_per_capita_day_2017"] /
    df_final["food_kcal_per_capita_day_2017"]
)

df_final["poultry_dependency_ratio"] = (
    df_final["poultry_import_1000_tonnes_2017"] /
    (df_final["poultry_import_1000_tonnes_2017"].median() + 1)
)

display(df_final.head().T)

,0,1,2,3,4
country,Afghanistan,Albania,Algeria,American Samoa,Andorra
population_2017,"36,296.113","2,884.169","41,389.189","5,447.900",77.001
population_cagr_2000_2017,0.033,-0.005,0.017,0.012,0.010
gdp_per_capita_2017,530.150,"4,521.752","4,134.936","6,450.320","40,632.206"
gdp_pc_growth_2017,1.902,9.831,4.228,6.002,1.825
political_stability_2017,-2.800,0.380,-0.920,1.220,1.420
food_kcal_per_capita_day_2017,"1,997.000","3,400.000","3,345.000","2,871.500","2,871.500"
food_protein_g_per_capita_day_2017,54.090,119.500,92.850,81.390,81.390
poultry_kcal_per_capita_day_2017,5.000,85.000,22.000,64.000,64.000
poultry_import_1000_tonnes_2017,29.000,38.000,2.000,16.000,16.000


In [45]:
candidate_final_vars = [
    "country",
    "population_2017",
    "population_cagr_2000_2017",
    "gdp_per_capita_2017",
    "gdp_pc_growth_2017",
    "political_stability_2017",
    "protein_kcal_ratio",
    "poultry_kcal_per_capita_day_2017",
    "poultry_import_1000_tonnes_2017",
]

df_final_model = df_final[candidate_final_vars].copy()

print("Final model dataset shape:", df_final_model.shape)
print("Number of countries:", df_final_model["country"].nunique())
display(df_final_model.head())

Final model dataset shape: (309, 9)
Number of countries: 309


,country,population_2017,population_cagr_2000_2017,gdp_per_capita_2017,gdp_pc_growth_2017,political_stability_2017,protein_kcal_ratio,poultry_kcal_per_capita_day_2017,poultry_import_1000_tonnes_2017
0,Afghanistan,"36,296.113",0.033,530.150,1.902,-2.800,0.027,5.000,29.000
1,Albania,"2,884.169",-0.005,"4,521.752",9.831,0.380,0.035,85.000,38.000
2,Algeria,"41,389.189",0.017,"4,134.936",4.228,-0.920,0.028,22.000,2.000
3,American Samoa,"5,447.900",0.012,"6,450.320",6.002,1.220,0.028,64.000,16.000
4,Andorra,77.001,0.010,"40,632.206",1.825,1.420,0.028,64.000,16.000


In [46]:
print("Missing values in final model dataset:")
display(df_final_model.isna().sum())

print("Duplicated countries:", df_final_model["country"].duplicated().sum())

print("Columns in final model:")
print(df_final_model.columns.tolist())

Missing values in final model dataset:


country                             0
population_2017                     0
population_cagr_2000_2017           0
gdp_per_capita_2017                 0
gdp_pc_growth_2017                  0
political_stability_2017            0
protein_kcal_ratio                  0
poultry_kcal_per_capita_day_2017    0
poultry_import_1000_tonnes_2017     0
dtype: int64

Duplicated countries: 0
Columns in final model:
['country', 'population_2017', 'population_cagr_2000_2017', 'gdp_per_capita_2017', 'gdp_pc_growth_2017', 'political_stability_2017', 'protein_kcal_ratio', 'poultry_kcal_per_capita_day_2017', 'poultry_import_1000_tonnes_2017']


In [47]:
df_final_model.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
country,309,309,Afghanistan,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
population_2017,309.000,NaN,NaN,NaN,"25,674.451","115,747.467",0.793,"1,384.059","5,447.900","11,339.254","1,421,021.791"
population_cagr_2000_2017,309.000,NaN,NaN,NaN,0.013,0.012,-0.012,0.006,0.012,0.017,0.094
gdp_per_capita_2017,309.000,NaN,NaN,NaN,"13,576.326","21,651.322",295.621,"4,041.995","6,450.320","11,020.663","173,611.815"
gdp_pc_growth_2017,309.000,NaN,NaN,NaN,5.909,6.170,-29.218,4.514,6.002,7.249,32.525
political_stability_2017,309.000,NaN,NaN,NaN,-0.093,0.766,-2.940,-0.250,-0.045,0.160,1.920
protein_kcal_ratio,309.000,NaN,NaN,NaN,0.028,0.003,0.019,0.028,0.028,0.029,0.041
poultry_kcal_per_capita_day_2017,309.000,NaN,NaN,NaN,69.877,45.437,0.000,56.000,64.000,70.000,243.000
poultry_import_1000_tonnes_2017,309.000,NaN,NaN,NaN,56.453,143.046,0.000,14.000,16.000,21.000,"1,069.000"


In [48]:
print("Missing values in final model dataset:")
display(df_final_model.isna().sum())

print("Duplicated countries:", df_final_model["country"].duplicated().sum())

Missing values in final model dataset:


country                             0
population_2017                     0
population_cagr_2000_2017           0
gdp_per_capita_2017                 0
gdp_pc_growth_2017                  0
political_stability_2017            0
protein_kcal_ratio                  0
poultry_kcal_per_capita_day_2017    0
poultry_import_1000_tonnes_2017     0
dtype: int64

Duplicated countries: 0


In [ ]:
output_dir = Path("../data_processed")
output_dir.mkdir(parents=True, exist_ok=True)

df_merged.to_csv(output_dir / "dataset_2017_merged_main_v2.csv", index=False)
df_merged_with_under.to_csv(output_dir / "dataset_2017_merged_with_under_optional_v2.csv", index=False)
df_working_clean.to_csv(output_dir / "dataset_2017_working_clean_v2.csv", index=False)
df_working_imputed.to_csv(output_dir / "dataset_2017_working_imputed_v2.csv", index=False)
df_final_model.to_csv(output_dir / "dataset_2017_final_model_v2.csv", index=False)

print("Saved files:")
print("-", output_dir / "dataset_2017_merged_main_v2.csv")
print("-", output_dir / "dataset_2017_merged_with_under_optional_v2.csv")
print("-", output_dir / "dataset_2017_working_clean_v2.csv")
print("-", output_dir / "dataset_2017_working_imputed_v2.csv")
print("-", output_dir / "dataset_2017_final_model_v2.csv")